In [ ]:
import os, math, warnings, numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

base = Path("/content")
files = {
    "Week_1": base / "Week1.xlsx",
    "Week_2": base / "Week2.xlsx",
    "Week_3": base / "Week3.xlsx",
    "Week_4": base / "Week4.xlsx",
}

missing = [str(p) for p in files.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing files: " + ", ".join(missing))

outdir = base / "hemp_analysis_output_lean"
outdir.mkdir(parents=True, exist_ok=True)

rename_map = {
    "Yield (kg per ha)": "Yield_kg_per_ha",
    "Yield kg per ha": "Yield_kg_per_ha",
    "Yieldkgperha": "Yield_kg_per_ha",
    "Mean_Canopy Cover": "Mean_Canopy_Cover",
    "Mean Canopy Cover": "Mean_Canopy_Cover",
    "MeanCanopyCover": "Mean_Canopy_Cover",
    "Mean_Height": "Mean_Height",
    "Mean Height": "Mean_Height",
    "MeanHeight": "Mean_Height",
    "Mean_SPAD": "Mean_SPAD",
    "Mean SPAD": "Mean_SPAD",
    "MeanSPAD": "Mean_SPAD",
    "MEANTD": "MEAN_TD",
    "Mean_TD": "MEAN_TD",
    "MEAN_TD": "MEAN_TD"
}

dfs = []
for wk, dat in zip(files.keys(), ["45 DAT", "60 DAT", "75 DAT", "90 DAT"]):
    df = pd.read_excel(files[wk])
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})
    df["Week"] = wk
    df["DAT"] = dat
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data.columns = [c.strip() for c in data.columns]

for c in data.columns:
    if c not in ["Week", "DAT"]:
        data[c] = pd.to_numeric(data[c], errors="coerce")

data.to_csv(outdir / "cleaned_combined_data.csv", index=False)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneOut, KFold, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from scipy.stats import loguniform, randint

traits = {
    "Yield_kg_per_ha": "Yield",
    "Mean_SPAD": "SPAD",
    "Mean_Height": "Height",
    "Mean_Canopy_Cover": "Canopy Cover"
}

predictors = ["VARI", "TGI", "SIP2", "NDVI", "NDRE", "MCARI", "LCI", "GNDVI", "BNDVI", "MEAN_TD"]

model_specs = {
    "Ridge": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge())
        ]),
        "dist": {"model__alpha": loguniform(0.01, 100.0)},
        "n_iter": 6
    },
    "PLSR": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PLSRegression())
        ]),
        "dist": {"model__n_components": randint(1, 6)},
        "n_iter": 5
    },
    "SVR": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVR(kernel="rbf"))
        ]),
        "dist": {
            "model__C": loguniform(0.1, 100.0),
            "model__gamma": ["scale", "auto", 0.01, 0.1],
            "model__epsilon": loguniform(0.01, 0.2)
        },
        "n_iter": 6
    }
}

def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def fit_predict_nested_loocv(X, y, spec, seed=42, inner_k=3):
    loo = LeaveOneOut()
    preds = np.full(len(y), np.nan, dtype=float)
    for tr, te in loo.split(X):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr = y.iloc[tr]
        inner_cv = KFold(n_splits=inner_k, shuffle=True, random_state=seed) if len(ytr) > inner_k else LeaveOneOut()
        search = RandomizedSearchCV(
            spec["pipe"],
            spec["dist"],
            n_iter=spec["n_iter"],
            scoring="r2",
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
            random_state=seed,
            error_score="raise"
        )
        search.fit(Xtr, ytr)
        preds[te[0]] = search.predict(Xte)[0]
    return preds

weekly_rows = []
pooled_rows = []

for wk in files.keys():
    d0 = data[data["Week"] == wk].copy()
    print(f"Starting {wk}")
    for target_col, trait in traits.items():
        if target_col not in d0.columns:
            continue
        d = d0[predictors + [target_col]].dropna().copy()
        if len(d) < 6:
            continue
        X, y = d[predictors], d[target_col]
        print(f"  Target: {trait} | N={len(d)}")
        for model_name, spec in model_specs.items():
            print(f"    Model: {model_name}")
            preds = fit_predict_nested_loocv(X, y, spec, seed=42, inner_k=3)
            weekly_rows.append({
                "Week": wk,
                "DAT": d0["DAT"].iloc[0],
                "Trait": trait,
                "Target": target_col,
                "Model": model_name,
                "N": len(d),
                "R2_LOOCV_Nested": r2_score(y, preds),
                "RMSE_LOOCV_Nested": rmse(y, preds),
                "MAE_LOOCV_Nested": mean_absolute_error(y, preds)
            })

for target_col, trait in traits.items():
    if target_col not in data.columns:
        continue
    d = data[predictors + [target_col]].dropna().copy()
    if len(d) < 12:
        continue
    X, y = d[predictors], d[target_col]
    print(f"Pooled target: {trait} | N={len(d)}")
    for model_name in ["Ridge", "PLSR", "SVR"]:
        print(f"  Pooled model: {model_name}")
        spec = model_specs[model_name]
        preds = fit_predict_nested_loocv(X, y, spec, seed=42, inner_k=3)
        pooled_rows.append({
            "Scope": "Pooled_All_Weeks",
            "Trait": trait,
            "Target": target_col,
            "Model": model_name,
            "N": len(d),
            "R2_LOOCV_Nested": r2_score(y, preds),
            "RMSE_LOOCV_Nested": rmse(y, preds),
            "MAE_LOOCV_Nested": mean_absolute_error(y, preds)
        })

weekly_df = pd.DataFrame(weekly_rows)
pooled_df = pd.DataFrame(pooled_rows)

weekly_df.to_csv(outdir / "weekly_nested_cv_results.csv", index=False)
pooled_df.to_csv(outdir / "pooled_nested_cv_results.csv", index=False)

weekly_summary = weekly_df.groupby(["Trait", "Model"], as_index=False).agg(
    N=("N", "mean"),
    R2_mean=("R2_LOOCV_Nested", "mean"),
    RMSE_mean=("RMSE_LOOCV_Nested", "mean"),
    MAE_mean=("MAE_LOOCV_Nested", "mean")
).sort_values(["Trait", "R2_mean"], ascending=[True, False])

pooled_summary = pooled_df.groupby(["Trait", "Model"], as_index=False).agg(
    N=("N", "mean"),
    R2_mean=("R2_LOOCV_Nested", "mean"),
    RMSE_mean=("RMSE_LOOCV_Nested", "mean"),
    MAE_mean=("MAE_LOOCV_Nested", "mean")
).sort_values(["Trait", "R2_mean"], ascending=[True, False])

weekly_summary.to_csv(outdir / "weekly_model_summary.csv", index=False)
pooled_summary.to_csv(outdir / "pooled_model_summary.csv", index=False)

best_weekly = weekly_df.sort_values(["Trait", "R2_LOOCV_Nested"], ascending=[True, False]).groupby("Trait").head(1)
best_pooled = pooled_df.sort_values(["Trait", "R2_LOOCV_Nested"], ascending=[True, False]).groupby("Trait").head(1)
best_weekly.to_csv(outdir / "best_weekly_model_by_trait.csv", index=False)
best_pooled.to_csv(outdir / "best_pooled_model_by_trait.csv", index=False)

print("DONE LEAN VERSION")
print(outdir)
print("\nWeekly summary:")
print(weekly_summary.to_string(index=False))
print("\nPooled summary:")
print(pooled_summary.to_string(index=False))

Starting Week_1
  Target: Yield | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
  Target: SPAD | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
  Target: Height | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
  Target: Canopy Cover | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
Starting Week_2
  Target: Yield | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
  Target: SPAD | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
  Target: Height | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
  Target: Canopy Cover | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
Starting Week_3
  Target: Yield | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
  Target: SPAD | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
  Target: Height | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
  Target: Canopy Cover | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
Starting Week_4
  Target: Yield | N=18
    Model: Ridge
    Model: PLSR
    Mod

In [ ]:
import os, glob
print(glob.glob('/content/*.xlsx'))

['/content/Week1.xlsx', '/content/Week3.xlsx', '/content/Week2.xlsx', '/content/Week4.xlsx']


In [ ]:
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor

In [ ]:
model_specs = {
    "Ridge": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge())
        ]),
        "dist": {"model__alpha": loguniform(0.01, 100.0)},
        "n_iter": 6
    },
    "PLSR": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PLSRegression())
        ]),
        "dist": {"model__n_components": randint(1, 6)},
        "n_iter": 5
    },
    "SVR": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVR(kernel="rbf"))
        ]),
        "dist": {
            "model__C": loguniform(0.1, 100.0),
            "model__gamma": ["scale", "auto", 0.01, 0.1],
            "model__epsilon": loguniform(0.01, 0.2)
        },
        "n_iter": 6
    },
    "RF": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
        ]),
        "dist": {
            "model__n_estimators": randint(100, 501),
            "model__max_depth": [None, 3, 5, 8, 12],
            "model__min_samples_split": randint(2, 8),
            "model__min_samples_leaf": randint(1, 5),
            "model__max_features": ["sqrt", "log2", 0.5, 0.8]
        },
        "n_iter": 6
    },
    "GB": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingRegressor(random_state=42))
        ]),
        "dist": {
            "model__n_estimators": randint(50, 301),
            "model__learning_rate": loguniform(0.01, 0.3),
            "model__max_depth": randint(1, 5),
            "model__min_samples_split": randint(2, 10),
            "model__min_samples_leaf": randint(1, 5),
            "model__subsample": [0.6, 0.8, 1.0]
        },
        "n_iter": 6
    }
}

In [ ]:
import os, math, warnings, numpy as np, pandas as pd
from pathlib import Path
warnings.filterwarnings("ignore")

base = Path("/content")
files = {
    "Week_1": base / "Week1.xlsx",
    "Week_2": base / "Week2.xlsx",
    "Week_3": base / "Week3.xlsx",
    "Week_4": base / "Week4.xlsx",
}

missing = [str(p) for p in files.values() if not p.exists()]
if missing:
    raise FileNotFoundError("Missing files: " + ", ".join(missing))

outdir = base / "hemp_analysis_output_lean"
outdir.mkdir(parents=True, exist_ok=True)

rename_map = {
    "Yield (kg per ha)": "Yield_kg_per_ha",
    "Yield kg per ha": "Yield_kg_per_ha",
    "Yieldkgperha": "Yield_kg_per_ha",
    "Mean_Canopy Cover": "Mean_Canopy_Cover",
    "Mean Canopy Cover": "Mean_Canopy_Cover",
    "MeanCanopyCover": "Mean_Canopy_Cover",
    "Mean_Height": "Mean_Height",
    "Mean Height": "Mean_Height",
    "MeanHeight": "Mean_Height",
    "Mean_SPAD": "Mean_SPAD",
    "Mean SPAD": "Mean_SPAD",
    "MeanSPAD": "Mean_SPAD",
    "MEANTD": "MEAN_TD",
    "Mean_TD": "MEAN_TD",
    "MEAN_TD": "MEAN_TD"
}

dfs = []
for wk, dat in zip(files.keys(), ["45 DAT", "60 DAT", "75 DAT", "90 DAT"]):
    df = pd.read_excel(files[wk])
    df.columns = [str(c).strip() for c in df.columns]
    df = df.rename(columns={c: rename_map.get(c, c) for c in df.columns})
    df["Week"] = wk
    df["DAT"] = dat
    dfs.append(df)

data = pd.concat(dfs, ignore_index=True)
data.columns = [c.strip() for c in data.columns]

for c in data.columns:
    if c not in ["Week", "DAT"]:
        data[c] = pd.to_numeric(data[c], errors="coerce")

data.to_csv(outdir / "cleaned_combined_data.csv", index=False)

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import LeaveOneOut, KFold, RandomizedSearchCV
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from scipy.stats import loguniform, randint

traits = {
    "Yield_kg_per_ha": "Yield",
    "Mean_SPAD": "SPAD",
    "Mean_Height": "Height",
    "Mean_Canopy_Cover": "Canopy Cover"
}

predictors = ["VARI", "TGI", "SIP2", "NDVI", "NDRE", "MCARI", "LCI", "GNDVI", "BNDVI", "MEAN_TD"]

model_specs = {
    "Ridge": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", Ridge())
        ]),
        "dist": {"model__alpha": loguniform(0.01, 100.0)},
        "n_iter": 6
    },
    "PLSR": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", PLSRegression())
        ]),
        "dist": {"model__n_components": randint(1, 6)},
        "n_iter": 5
    },
    "SVR": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            ("model", SVR(kernel="rbf"))
        ]),
        "dist": {
            "model__C": loguniform(0.1, 100.0),
            "model__gamma": ["scale", "auto", 0.01, 0.1],
            "model__epsilon": loguniform(0.01, 0.2)
        },
        "n_iter": 6
    },
    "RF": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", RandomForestRegressor(random_state=42, n_jobs=-1))
        ]),
        "dist": {
            "model__n_estimators": randint(100, 501),
            "model__max_depth": [None, 3, 5, 8, 12],
            "model__min_samples_split": randint(2, 8),
            "model__min_samples_leaf": randint(1, 5),
            "model__max_features": ["sqrt", "log2", 0.5, 0.8]
        },
        "n_iter": 6
    },
    "GB": {
        "pipe": Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("model", GradientBoostingRegressor(random_state=42))
        ]),
        "dist": {
            "model__n_estimators": randint(50, 301),
            "model__learning_rate": loguniform(0.01, 0.3),
            "model__max_depth": randint(1, 5),
            "model__min_samples_split": randint(2, 10),
            "model__min_samples_leaf": randint(1, 5),
            "model__subsample": [0.6, 0.8, 1.0]
        },
        "n_iter": 6
    }
}

def rmse(y_true, y_pred):
    return math.sqrt(mean_squared_error(y_true, y_pred))

def fit_predict_nested_loocv(X, y, spec, seed=42, inner_k=3):
    loo = LeaveOneOut()
    preds = np.full(len(y), np.nan, dtype=float)
    for tr, te in loo.split(X):
        Xtr, Xte = X.iloc[tr], X.iloc[te]
        ytr = y.iloc[tr]
        inner_cv = KFold(n_splits=inner_k, shuffle=True, random_state=seed) if len(ytr) > inner_k else LeaveOneOut()
        search = RandomizedSearchCV(
            spec["pipe"],
            spec["dist"],
            n_iter=spec["n_iter"],
            scoring="r2",
            cv=inner_cv,
            n_jobs=-1,
            refit=True,
            random_state=seed,
            error_score="raise"
        )
        search.fit(Xtr, ytr)
        preds[te[0]] = search.predict(Xte)[0]
    return preds

weekly_rows = []
pooled_rows = []

for wk in files.keys():
    d0 = data[data["Week"] == wk].copy()
    print(f"Starting {wk}")
    for target_col, trait in traits.items():
        if target_col not in d0.columns:
            continue
        d = d0[predictors + [target_col]].dropna().copy()
        if len(d) < 6:
            continue
        X, y = d[predictors], d[target_col]
        print(f"  Target: {trait} | N={len(d)}")
        for model_name, spec in model_specs.items():
            print(f"    Model: {model_name}")
            preds = fit_predict_nested_loocv(X, y, spec, seed=42, inner_k=3)
            weekly_rows.append({
                "Week": wk,
                "DAT": d0["DAT"].iloc[0],
                "Trait": trait,
                "Target": target_col,
                "Model": model_name,
                "N": len(d),
                "R2_LOOCV_Nested": r2_score(y, preds),
                "RMSE_LOOCV_Nested": rmse(y, preds),
                "MAE_LOOCV_Nested": mean_absolute_error(y, preds)
            })

for target_col, trait in traits.items():
    if target_col not in data.columns:
        continue
    d = data[predictors + [target_col]].dropna().copy()
    if len(d) < 12:
        continue
    X, y = d[predictors], d[target_col]
    print(f"Pooled target: {trait} | N={len(d)}")
    for model_name in ["Ridge", "PLSR", "SVR", "RF", "GB"]:
        print(f"  Pooled model: {model_name}")
        spec = model_specs[model_name]
        preds = fit_predict_nested_loocv(X, y, spec, seed=42, inner_k=3)
        pooled_rows.append({
            "Scope": "Pooled_All_Weeks",
            "Trait": trait,
            "Target": target_col,
            "Model": model_name,
            "N": len(d),
            "R2_LOOCV_Nested": r2_score(y, preds),
            "RMSE_LOOCV_Nested": rmse(y, preds),
            "MAE_LOOCV_Nested": mean_absolute_error(y, preds)
        })

weekly_df = pd.DataFrame(weekly_rows)
pooled_df = pd.DataFrame(pooled_rows)

weekly_df.to_csv(outdir / "weekly_nested_cv_results.csv", index=False)
pooled_df.to_csv(outdir / "pooled_nested_cv_results.csv", index=False)

weekly_summary = weekly_df.groupby(["Trait", "Model"], as_index=False).agg(
    N=("N", "mean"),
    R2_mean=("R2_LOOCV_Nested", "mean"),
    RMSE_mean=("RMSE_LOOCV_Nested", "mean"),
    MAE_mean=("MAE_LOOCV_Nested", "mean")
).sort_values(["Trait", "R2_mean"], ascending=[True, False])

pooled_summary = pooled_df.groupby(["Trait", "Model"], as_index=False).agg(
    N=("N", "mean"),
    R2_mean=("R2_LOOCV_Nested", "mean"),
    RMSE_mean=("RMSE_LOOCV_Nested", "mean"),
    MAE_mean=("MAE_LOOCV_Nested", "mean")
).sort_values(["Trait", "R2_mean"], ascending=[True, False])

weekly_summary.to_csv(outdir / "weekly_model_summary.csv", index=False)
pooled_summary.to_csv(outdir / "pooled_model_summary.csv", index=False)

best_weekly = weekly_df.sort_values(["Trait", "R2_LOOCV_Nested"], ascending=[True, False]).groupby("Trait").head(1)
best_pooled = pooled_df.sort_values(["Trait", "R2_LOOCV_Nested"], ascending=[True, False]).groupby("Trait").head(1)
best_weekly.to_csv(outdir / "best_weekly_model_by_trait.csv", index=False)
best_pooled.to_csv(outdir / "best_pooled_model_by_trait.csv", index=False)

print("DONE LEAN VERSION")
print(outdir)
print("\nWeekly summary:")
print(weekly_summary.to_string(index=False))
print("\nPooled summary:")
print(pooled_summary.to_string(index=False))

Starting Week_1
  Target: Yield | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
    Model: RF
    Model: GB
  Target: SPAD | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
    Model: RF
    Model: GB
  Target: Height | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
    Model: RF
    Model: GB
  Target: Canopy Cover | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
    Model: RF
    Model: GB
Starting Week_2
  Target: Yield | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
    Model: RF
    Model: GB
  Target: SPAD | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
    Model: RF
    Model: GB
  Target: Height | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
    Model: RF
    Model: GB
  Target: Canopy Cover | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
    Model: RF
    Model: GB
Starting Week_3
  Target: Yield | N=18
    Model: Ridge
    Model: PLSR
    Model: SVR
    Model: RF
    Model: GB
  Target: SPAD | N=18
    Model: Ridge
    Mode

In [ ]:
import os, math, warnings, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

outdir = Path("/content/hemp_analysis_output_lean")
outdir.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="white", font_scale=1.1)

model_order = ["Ridge", "PLSR", "RF", "GB", "SVR"]
trait_order = ["Yield", "SPAD", "Height", "Canopy Cover"]
week_order = ["45 DAT", "60 DAT", "75 DAT", "90 DAT"]

def _style_axes(ax):
    ax.grid(False)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)

# =========================================================
# FIGURE 7: HEATMAPS OF MODEL PERFORMANCE
# =========================================================
week_summary = weekly_df.copy()
week_summary["Model"] = pd.Categorical(week_summary["Model"], categories=model_order, ordered=True)
week_summary["Trait"] = pd.Categorical(week_summary["Trait"], categories=trait_order, ordered=True)
week_summary["DAT"] = pd.Categorical(week_summary["DAT"], categories=week_order, ordered=True)

r2_mat = week_summary.pivot_table(index="Model", columns="Trait", values="R2_LOOCV_Nested", aggfunc="mean").reindex(index=model_order, columns=trait_order)
rmse_mat = week_summary.pivot_table(index="Model", columns="Trait", values="RMSE_LOOCV_Nested", aggfunc="mean").reindex(index=model_order, columns=trait_order)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

sns.heatmap(
    r2_mat,
    ax=axes[0],
    cmap="Blues",
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Mean R²"}
)
axes[0].set_title("Figure 7A. Mean R² across models and traits", fontweight="bold")
axes[0].set_xlabel("")
axes[0].set_ylabel("Model")
_style_axes(axes[0])

sns.heatmap(
    rmse_mat,
    ax=axes[1],
    cmap="YlOrBr",
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Mean RMSE"}
)
axes[1].set_title("Figure 7B. Mean RMSE across models and traits", fontweight="bold")
axes[1].set_xlabel("")
axes[1].set_ylabel("Model")
_style_axes(axes[1])

plt.tight_layout()
plt.savefig(outdir / "Figure7_Model_Performance_Heatmaps.png", dpi=300, bbox_inches="tight")
plt.close()

# =========================================================
# FIGURE 8: COMPOSITE PREDICTABILITY INDEX
# Define it as normalized R² minus normalized RMSE so
# higher is better and error is penalized.
# =========================================================
summary = week_summary.groupby(["Trait", "Model"], as_index=False).agg(
    R2_mean=("R2_LOOCV_Nested", "mean"),
    RMSE_mean=("RMSE_LOOCV_Nested", "mean"),
    MAE_mean=("MAE_LOOCV_Nested", "mean"),
    N=("N", "mean")
)

summary["R2_z"] = summary.groupby("Trait")["R2_mean"].transform(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9))
summary["RMSE_z"] = summary.groupby("Trait")["RMSE_mean"].transform(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9))
summary["Composite_Predictability"] = summary["R2_z"] - summary["RMSE_z"]

comp_mat = summary.pivot(index="Model", columns="Trait", values="Composite_Predictability").reindex(index=model_order, columns=trait_order)

plt.figure(figsize=(9, 6))
ax = sns.heatmap(
    comp_mat,
    cmap="RdYlGn",
    center=0,
    annot=True,
    fmt=".2f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Composite Predictability"}
)
plt.title("Figure 8. Composite Predictability Index", fontweight="bold")
plt.xlabel("")
plt.ylabel("Model")
_style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure8_Composite_Predictability.png", dpi=300, bbox_inches="tight")
plt.close()

summary.to_csv(outdir / "Figure8_Composite_Predictability_Table.csv", index=False)

# =========================================================
# HELPERS FOR FEATURE IMPORTANCE
# =========================================================
feature_cols = ["VARI", "TGI", "SIP2", "NDVI", "NDRE", "MCARI", "LCI", "GNDVI", "BNDVI", "MEAN_TD"]
target_map = {
    "Yield_kg_per_ha": "Yield",
    "Mean_SPAD": "SPAD",
    "Mean_Height": "Height",
    "Mean_Canopy_Cover": "Canopy Cover"
}

def get_xy(target_col):
    d = data[feature_cols + [target_col]].dropna().copy()
    X = d[feature_cols]
    y = d[target_col]
    return X, y

# =========================================================
# FIGURE 9: RANDOM FOREST FEATURE IMPORTANCE
# =========================================================
rf_imp_rows = []
for target_col, trait_name in target_map.items():
    if target_col not in data.columns:
        continue
    X, y = get_xy(target_col)
    if len(X) < 5:
        continue
    rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    for f, v in zip(feature_cols, rf.feature_importances_):
        rf_imp_rows.append([trait_name, f, v])

rf_imp = pd.DataFrame(rf_imp_rows, columns=["Trait", "Feature", "Importance"])
rf_pivot = rf_imp.pivot_table(index="Feature", columns="Trait", values="Importance", aggfunc="mean")

plt.figure(figsize=(10, 6))
ax = sns.heatmap(
    rf_pivot,
    cmap="Blues",
    annot=True,
    fmt=".3f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Importance"}
)
plt.title("Figure 9. Feature Importance (Random Forest)", fontweight="bold")
plt.xlabel("")
plt.ylabel("Feature")
_style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure9_RF_Feature_Importance.png", dpi=300, bbox_inches="tight")
plt.close()

rf_imp.to_csv(outdir / "Figure9_RF_Importance.csv", index=False)

# =========================================================
# FIGURE 10: RIDGE COEFFICIENT IMPORTANCE
# Use absolute standardized coefficients
# =========================================================
ridge_rows = []
for target_col, trait_name in target_map.items():
    if target_col not in data.columns:
        continue
    d = data[feature_cols + [target_col]].dropna().copy()
    if len(d) < 5:
        continue
    X = d[feature_cols]
    y = d[target_col]
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ])
    pipe.fit(X, y)
    coef = np.abs(pipe.named_steps["model"].coef_)
    coef = coef / coef.sum() if coef.sum() != 0 else coef
    for f, v in zip(feature_cols, coef):
        ridge_rows.append([trait_name, f, v])

ridge_imp = pd.DataFrame(ridge_rows, columns=["Trait", "Feature", "Importance"])
ridge_pivot = ridge_imp.pivot_table(index="Feature", columns="Trait", values="Importance", aggfunc="mean")

plt.figure(figsize=(10, 6))
ax = sns.heatmap(
    ridge_pivot,
    cmap="YlGnBu",
    annot=True,
    fmt=".3f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Importance"}
)
plt.title("Figure 10. Feature Importance (Ridge Regression)", fontweight="bold")
plt.xlabel("")
plt.ylabel("Feature")
_style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure10_Ridge_Feature_Importance.png", dpi=300, bbox_inches="tight")
plt.close()

ridge_imp.to_csv(outdir / "Figure10_Ridge_Importance.csv", index=False)

# =========================================================
# FIGURE 11: GRADIENT BOOSTING SHAP IMPORTANCE
# =========================================================
shap_rows = []
for target_col, trait_name in target_map.items():
    if target_col not in data.columns:
        continue
    d = data[feature_cols + [target_col]].dropna().copy()
    if len(d) < 5:
        continue
    X = d[feature_cols]
    y = d[target_col]
    gb = GradientBoostingRegressor(random_state=42)
    gb.fit(X, y)
    explainer = shap.TreeExplainer(gb)
    shap_vals = explainer.shap_values(X)
    shap_vals = np.array(shap_vals)
    imp = np.abs(shap_vals).mean(axis=0)
    imp = imp / imp.sum() if imp.sum() != 0 else imp
    for f, v in zip(feature_cols, imp):
        shap_rows.append([trait_name, f, v])

shap_imp = pd.DataFrame(shap_rows, columns=["Trait", "Feature", "Importance"])
shap_pivot = shap_imp.pivot_table(index="Feature", columns="Trait", values="Importance", aggfunc="mean")

plt.figure(figsize=(10, 6))
ax = sns.heatmap(
    shap_pivot,
    cmap="magma",
    annot=True,
    fmt=".3f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Mean |SHAP|"}
)
plt.title("Figure 11. Feature Importance (Gradient Boosting SHAP Importance)", fontweight="bold")
plt.xlabel("")
plt.ylabel("Feature")
_style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure11_GB_SHAP_Importance.png", dpi=300, bbox_inches="tight")
plt.close()

shap_imp.to_csv(outdir / "Figure11_GB_SHAP_Importance.csv", index=False)

# =========================================================
# FIGURE 12: PLSR VIP IMPORTANCE
# =========================================================
def vip_scores(pls, X, y):
    T = pls.x_scores_
    W = pls.x_weights_
    Q = pls.y_loadings_
    p, h = W.shape
    s = np.diag(T.T @ T @ Q.T @ Q).reshape(h, -1)
    total_s = s.sum()
    vip = np.sqrt(p * (W**2 @ s) / (total_s + 1e-12)).ravel()
    return vip

plsr_rows = []
for target_col, trait_name in target_map.items():
    if target_col not in data.columns:
        continue
    d = data[feature_cols + [target_col]].dropna().copy()
    if len(d) < 5:
        continue
    X = d[feature_cols].values
    y = d[target_col].values.reshape(-1, 1)
    ncomp = min(5, X.shape[1], max(1, X.shape[0] - 1))
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", PLSRegression(n_components=ncomp))
    ])
    pipe.fit(X, y)
    pls = pipe.named_steps["model"]
    vip = vip_scores(pls, X, y)
    vip = vip / vip.sum() if vip.sum() != 0 else vip
    for f, v in zip(feature_cols, vip):
        plsr_rows.append([trait_name, f, v])

plsr_imp = pd.DataFrame(plsr_rows, columns=["Trait", "Feature", "Importance"])
plsr_pivot = plsr_imp.pivot_table(index="Feature", columns="Trait", values="Importance", aggfunc="mean")

plt.figure(figsize=(10, 6))
ax = sns.heatmap(
    plsr_pivot,
    cmap="YlOrBr",
    annot=True,
    fmt=".3f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "VIP"}
)
plt.title("Figure 12. PLSR Importance (VIP)", fontweight="bold")
plt.xlabel("")
plt.ylabel("Feature")
_style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure12_PLSR_VIP.png", dpi=300, bbox_inches="tight")
plt.close()

plsr_imp.to_csv(outdir / "Figure12_PLSR_VIP.csv", index=False)

# =========================================================
# FIGURE 13: SVR PERMUTATION IMPORTANCE
# =========================================================
perm_rows = []
for target_col, trait_name in target_map.items():
    if target_col not in data.columns:
        continue
    d = data[feature_cols + [target_col]].dropna().copy()
    if len(d) < 5:
        continue
    X = d[feature_cols].values
    y = d[target_col].values
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf", C=10, epsilon=0.1))
    ])
    pipe.fit(X, y)
    perm = permutation_importance(pipe, X, y, n_repeats=20, random_state=42, scoring="r2")
    vals = perm.importances_mean
    vals = vals / np.sum(np.abs(vals)) if np.sum(np.abs(vals)) != 0 else vals
    for f, v in zip(feature_cols, vals):
        perm_rows.append([trait_name, f, v])

perm_imp = pd.DataFrame(perm_rows, columns=["Trait", "Feature", "Importance"])
perm_pivot = perm_imp.pivot_table(index="Feature", columns="Trait", values="Importance", aggfunc="mean")

plt.figure(figsize=(10, 6))
ax = sns.heatmap(
    perm_pivot,
    cmap="coolwarm",
    center=0,
    annot=True,
    fmt=".3f",
    linewidths=0.5,
    linecolor="white",
    cbar_kws={"label": "Permutation Importance"}
)
plt.title("Figure 13. SVR Permutation-Based Feature Importance", fontweight="bold")
plt.xlabel("")
plt.ylabel("Feature")
_style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure13_SVR_Permutation_Importance.png", dpi=300, bbox_inches="tight")
plt.close()

perm_imp.to_csv(outdir / "Figure13_SVR_Permutation_Importance.csv", index=False)

# Save combined outputs
rf_pivot.to_csv(outdir / "Figure9_RF_pivot.csv")
ridge_pivot.to_csv(outdir / "Figure10_Ridge_pivot.csv")
shap_pivot.to_csv(outdir / "Figure11_GB_SHAP_pivot.csv")
plsr_pivot.to_csv(outdir / "Figure12_PLSR_VIP_pivot.csv")
perm_pivot.to_csv(outdir / "Figure13_SVR_Permutation_pivot.csv")

print("Figures 7–13 created in:", outdir)

Figures 7–13 created in: /content/hemp_analysis_output_lean


In [ ]:
# ============================================================
# HEMP PHENOME — DOWNLOAD ALL OUTPUTS
# Paste this as a NEW cell at the END of your Colab notebook.
# Run it after all previous cells have finished.
# It will produce every figure + CSV, zip them, and trigger
# a single browser download.
# ============================================================

import os, math, warnings, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from google.colab import files

warnings.filterwarnings("ignore")

# ── Output folder ────────────────────────────────────────────────────────────
outdir = Path("/content/hemp_analysis_output_lean")
outdir.mkdir(parents=True, exist_ok=True)

# ── Styling helpers ──────────────────────────────────────────────────────────
sns.set_theme(style="white", font_scale=1.1)
model_order = ["Ridge", "PLSR", "RF", "GB", "SVR"]
trait_order  = ["Yield", "SPAD", "Height", "Canopy Cover"]
week_order   = ["45 DAT", "60 DAT", "75 DAT", "90 DAT"]
feature_cols = ["VARI", "TGI", "SIP2", "NDVI", "NDRE",
                "MCARI", "LCI", "GNDVI", "BNDVI", "MEAN_TD"]
target_map   = {
    "Yield_kg_per_ha":   "Yield",
    "Mean_SPAD":         "SPAD",
    "Mean_Height":       "Height",
    "Mean_Canopy_Cover": "Canopy Cover",
}

def _style_axes(ax):
    ax.grid(False)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)

def get_xy(target_col):
    d = data[feature_cols + [target_col]].dropna().copy()
    return d[feature_cols], d[target_col]

# ── Save existing CSV results (already computed) ─────────────────────────────
print("Saving CV result tables …")
weekly_df.to_csv(outdir / "weekly_nested_cv_results.csv", index=False)
pooled_df.to_csv(outdir / "pooled_nested_cv_results.csv", index=False)

weekly_summary = (
    weekly_df.groupby(["Trait", "Model"], as_index=False)
    .agg(N=("N","mean"), R2_mean=("R2_LOOCV_Nested","mean"),
         RMSE_mean=("RMSE_LOOCV_Nested","mean"), MAE_mean=("MAE_LOOCV_Nested","mean"))
    .sort_values(["Trait","R2_mean"], ascending=[True,False])
)
pooled_summary = (
    pooled_df.groupby(["Trait", "Model"], as_index=False)
    .agg(N=("N","mean"), R2_mean=("R2_LOOCV_Nested","mean"),
         RMSE_mean=("RMSE_LOOCV_Nested","mean"), MAE_mean=("MAE_LOOCV_Nested","mean"))
    .sort_values(["Trait","R2_mean"], ascending=[True,False])
)
weekly_summary.to_csv(outdir / "weekly_model_summary.csv", index=False)
pooled_summary.to_csv(outdir / "pooled_model_summary.csv", index=False)

best_weekly = (weekly_df.sort_values(["Trait","R2_LOOCV_Nested"], ascending=[True,False])
               .groupby("Trait").head(1))
best_pooled = (pooled_df.sort_values(["Trait","R2_LOOCV_Nested"], ascending=[True,False])
               .groupby("Trait").head(1))
best_weekly.to_csv(outdir / "best_weekly_model_by_trait.csv", index=False)
best_pooled.to_csv(outdir / "best_pooled_model_by_trait.csv", index=False)

# ── FIGURE 7: Model performance heatmaps ─────────────────────────────────────
print("Figure 7 …")
week_summary = weekly_df.copy()
r2_mat   = week_summary.pivot_table(index="Model", columns="Trait",
            values="R2_LOOCV_Nested",   aggfunc="mean").reindex(index=model_order, columns=trait_order)
rmse_mat = week_summary.pivot_table(index="Model", columns="Trait",
            values="RMSE_LOOCV_Nested", aggfunc="mean").reindex(index=model_order, columns=trait_order)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
sns.heatmap(r2_mat,   ax=axes[0], cmap="Blues",  annot=True, fmt=".2f",
            linewidths=0.5, linecolor="white", cbar_kws={"label":"Mean R²"})
axes[0].set_title("Figure 7A. Mean R² across models and traits", fontweight="bold")
axes[0].set_xlabel(""); axes[0].set_ylabel("Model"); _style_axes(axes[0])

sns.heatmap(rmse_mat, ax=axes[1], cmap="YlOrBr", annot=True, fmt=".2f",
            linewidths=0.5, linecolor="white", cbar_kws={"label":"Mean RMSE"})
axes[1].set_title("Figure 7B. Mean RMSE across models and traits", fontweight="bold")
axes[1].set_xlabel(""); axes[1].set_ylabel("Model"); _style_axes(axes[1])
plt.tight_layout()
plt.savefig(outdir / "Figure7_Model_Performance_Heatmaps.png", dpi=300, bbox_inches="tight")
plt.close()

# ── FIGURE 8: Composite Predictability Index ─────────────────────────────────
print("Figure 8 …")
summary = (week_summary.groupby(["Trait","Model"], as_index=False)
           .agg(R2_mean=("R2_LOOCV_Nested","mean"), RMSE_mean=("RMSE_LOOCV_Nested","mean"),
                MAE_mean=("MAE_LOOCV_Nested","mean"), N=("N","mean")))
summary["R2_z"]   = summary.groupby("Trait")["R2_mean"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9))
summary["RMSE_z"] = summary.groupby("Trait")["RMSE_mean"].transform(
    lambda x: (x - x.min()) / (x.max() - x.min() + 1e-9))
summary["Composite_Predictability"] = summary["R2_z"] - summary["RMSE_z"]
comp_mat = summary.pivot(index="Model", columns="Trait",
           values="Composite_Predictability").reindex(index=model_order, columns=trait_order)
plt.figure(figsize=(9, 6))
ax = sns.heatmap(comp_mat, cmap="RdYlGn", center=0, annot=True, fmt=".2f",
                 linewidths=0.5, linecolor="white", cbar_kws={"label":"Composite Predictability"})
plt.title("Figure 8. Composite Predictability Index", fontweight="bold")
plt.xlabel(""); plt.ylabel("Model"); _style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure8_Composite_Predictability.png", dpi=300, bbox_inches="tight")
plt.close()
summary.to_csv(outdir / "Figure8_Composite_Predictability_Table.csv", index=False)

# ── FIGURE 9: Random Forest feature importance ────────────────────────────────
print("Figure 9 — Random Forest …")
rf_imp_rows = []
for tc, tn in target_map.items():
    if tc not in data.columns: continue
    X, y = get_xy(tc)
    if len(X) < 5: continue
    rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    for f, v in zip(feature_cols, rf.feature_importances_):
        rf_imp_rows.append([tn, f, v])
rf_imp   = pd.DataFrame(rf_imp_rows, columns=["Trait","Feature","Importance"])
rf_pivot = rf_imp.pivot_table(index="Feature", columns="Trait",
                               values="Importance", aggfunc="mean")
plt.figure(figsize=(10, 6))
ax = sns.heatmap(rf_pivot, cmap="Blues", annot=True, fmt=".3f",
                 linewidths=0.5, linecolor="white", cbar_kws={"label":"Importance"})
plt.title("Figure 9. Feature Importance (Random Forest)", fontweight="bold")
plt.xlabel(""); plt.ylabel("Feature"); _style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure9_RF_Feature_Importance.png", dpi=300, bbox_inches="tight")
plt.close()
rf_imp.to_csv(outdir / "Figure9_RF_Importance.csv", index=False)
rf_pivot.to_csv(outdir / "Figure9_RF_pivot.csv")

# ── FIGURE 10: Ridge coefficient importance ───────────────────────────────────
print("Figure 10 — Ridge …")
ridge_rows = []
for tc, tn in target_map.items():
    if tc not in data.columns: continue
    d = data[feature_cols + [tc]].dropna().copy()
    if len(d) < 5: continue
    X, y = d[feature_cols], d[tc]
    pipe = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
    pipe.fit(X, y)
    coef = np.abs(pipe.named_steps["model"].coef_)
    coef = coef / coef.sum() if coef.sum() != 0 else coef
    for f, v in zip(feature_cols, coef):
        ridge_rows.append([tn, f, v])
ridge_imp   = pd.DataFrame(ridge_rows, columns=["Trait","Feature","Importance"])
ridge_pivot = ridge_imp.pivot_table(index="Feature", columns="Trait",
                                     values="Importance", aggfunc="mean")
plt.figure(figsize=(10, 6))
ax = sns.heatmap(ridge_pivot, cmap="YlGnBu", annot=True, fmt=".3f",
                 linewidths=0.5, linecolor="white", cbar_kws={"label":"Importance"})
plt.title("Figure 10. Feature Importance (Ridge Regression)", fontweight="bold")
plt.xlabel(""); plt.ylabel("Feature"); _style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure10_Ridge_Feature_Importance.png", dpi=300, bbox_inches="tight")
plt.close()
ridge_imp.to_csv(outdir / "Figure10_Ridge_Importance.csv", index=False)
ridge_pivot.to_csv(outdir / "Figure10_Ridge_pivot.csv")

# ── FIGURE 11: Gradient Boosting SHAP importance ─────────────────────────────
print("Figure 11 — GB SHAP …")
shap_rows = []
for tc, tn in target_map.items():
    if tc not in data.columns: continue
    d = data[feature_cols + [tc]].dropna().copy()
    if len(d) < 5: continue
    X, y = d[feature_cols], d[tc]
    gb = GradientBoostingRegressor(random_state=42)
    gb.fit(X, y)
    explainer = shap.TreeExplainer(gb)
    shap_vals = np.array(explainer.shap_values(X))
    imp = np.abs(shap_vals).mean(axis=0)
    imp = imp / imp.sum() if imp.sum() != 0 else imp
    for f, v in zip(feature_cols, imp):
        shap_rows.append([tn, f, v])
shap_imp   = pd.DataFrame(shap_rows, columns=["Trait","Feature","Importance"])
shap_pivot = shap_imp.pivot_table(index="Feature", columns="Trait",
                                   values="Importance", aggfunc="mean")
plt.figure(figsize=(10, 6))
ax = sns.heatmap(shap_pivot, cmap="magma", annot=True, fmt=".3f",
                 linewidths=0.5, linecolor="white", cbar_kws={"label":"Mean |SHAP|"})
plt.title("Figure 11. Feature Importance (Gradient Boosting SHAP)", fontweight="bold")
plt.xlabel(""); plt.ylabel("Feature"); _style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure11_GB_SHAP_Importance.png", dpi=300, bbox_inches="tight")
plt.close()
shap_imp.to_csv(outdir / "Figure11_GB_SHAP_Importance.csv", index=False)
shap_pivot.to_csv(outdir / "Figure11_GB_SHAP_pivot.csv")

# ── FIGURE 12: PLSR VIP importance ───────────────────────────────────────────
print("Figure 12 — PLSR VIP …")
def vip_scores(pls, X, y):
    T, W, Q = pls.x_scores_, pls.x_weights_, pls.y_loadings_
    p, h = W.shape
    s = np.diag(T.T @ T @ Q.T @ Q).reshape(h, -1)
    vip = np.sqrt(p * (W**2 @ s) / (s.sum() + 1e-12)).ravel()
    return vip

plsr_rows = []
for tc, tn in target_map.items():
    if tc not in data.columns: continue
    d = data[feature_cols + [tc]].dropna().copy()
    if len(d) < 5: continue
    X = d[feature_cols].values
    y = d[tc].values.reshape(-1, 1)
    ncomp = min(5, X.shape[1], max(1, X.shape[0] - 1))
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("model", PLSRegression(n_components=ncomp))])
    pipe.fit(X, y)
    vip = vip_scores(pipe.named_steps["model"], X, y)
    vip = vip / vip.sum() if vip.sum() != 0 else vip
    for f, v in zip(feature_cols, vip):
        plsr_rows.append([tn, f, v])
plsr_imp   = pd.DataFrame(plsr_rows, columns=["Trait","Feature","Importance"])
plsr_pivot = plsr_imp.pivot_table(index="Feature", columns="Trait",
                                   values="Importance", aggfunc="mean")
plt.figure(figsize=(10, 6))
ax = sns.heatmap(plsr_pivot, cmap="YlOrBr", annot=True, fmt=".3f",
                 linewidths=0.5, linecolor="white", cbar_kws={"label":"VIP"})
plt.title("Figure 12. PLSR Importance (VIP)", fontweight="bold")
plt.xlabel(""); plt.ylabel("Feature"); _style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure12_PLSR_VIP.png", dpi=300, bbox_inches="tight")
plt.close()
plsr_imp.to_csv(outdir / "Figure12_PLSR_VIP.csv", index=False)
plsr_pivot.to_csv(outdir / "Figure12_PLSR_VIP_pivot.csv")

# ── FIGURE 13: SVR permutation importance ────────────────────────────────────
print("Figure 13 — SVR permutation …")
perm_rows = []
for tc, tn in target_map.items():
    if tc not in data.columns: continue
    d = data[feature_cols + [tc]].dropna().copy()
    if len(d) < 5: continue
    X = d[feature_cols].values
    y = d[tc].values
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("model", SVR(kernel="rbf", C=10, epsilon=0.1))])
    pipe.fit(X, y)
    perm = permutation_importance(pipe, X, y, n_repeats=20, random_state=42, scoring="r2")
    vals = perm.importances_mean
    vals = vals / np.sum(np.abs(vals)) if np.sum(np.abs(vals)) != 0 else vals
    for f, v in zip(feature_cols, vals):
        perm_rows.append([tn, f, v])
perm_imp   = pd.DataFrame(perm_rows, columns=["Trait","Feature","Importance"])
perm_pivot = perm_imp.pivot_table(index="Feature", columns="Trait",
                                   values="Importance", aggfunc="mean")
plt.figure(figsize=(10, 6))
ax = sns.heatmap(perm_pivot, cmap="coolwarm", center=0, annot=True, fmt=".3f",
                 linewidths=0.5, linecolor="white", cbar_kws={"label":"Permutation Importance"})
plt.title("Figure 13. SVR Permutation-Based Feature Importance", fontweight="bold")
plt.xlabel(""); plt.ylabel("Feature"); _style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure13_SVR_Permutation_Importance.png", dpi=300, bbox_inches="tight")
plt.close()
perm_imp.to_csv(outdir / "Figure13_SVR_Permutation_Importance.csv", index=False)
perm_pivot.to_csv(outdir / "Figure13_SVR_Permutation_pivot.csv")

# ── ZIP everything and download ───────────────────────────────────────────────
print("\nZipping all outputs …")
zip_path = "/content/HEMP_all_outputs.zip"
all_files = sorted(outdir.iterdir())

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in all_files:
        if f.is_file():
            zf.write(f, arcname=f.name)

n = len([f for f in all_files if f.is_file()])
print(f"Zipped {n} files → {zip_path}")
print("\nFiles included:")
for f in sorted(all_files):
    if f.is_file():
        print(f"  {f.name}")

print("\nStarting download …")
files.download(zip_path)
print("Done! Check your browser downloads.")

Saving CV result tables …
Figure 7 …
Figure 8 …
Figure 9 — Random Forest …
Figure 10 — Ridge …
Figure 11 — GB SHAP …
Figure 12 — PLSR VIP …
Figure 13 — SVR permutation …

Zipping all outputs …
Zipped 25 files → /content/HEMP_all_outputs.zip

Files included:
  Figure10_Ridge_Feature_Importance.png
  Figure10_Ridge_Importance.csv
  Figure10_Ridge_pivot.csv
  Figure11_GB_SHAP_Importance.csv
  Figure11_GB_SHAP_Importance.png
  Figure11_GB_SHAP_pivot.csv
  Figure12_PLSR_VIP.csv
  Figure12_PLSR_VIP.png
  Figure12_PLSR_VIP_pivot.csv
  Figure13_SVR_Permutation_Importance.csv
  Figure13_SVR_Permutation_Importance.png
  Figure13_SVR_Permutation_pivot.csv
  Figure7_Model_Performance_Heatmaps.png
  Figure8_Composite_Predictability.png
  Figure8_Composite_Predictability_Table.csv
  Figure9_RF_Feature_Importance.png
  Figure9_RF_Importance.csv
  Figure9_RF_pivot.csv
  best_pooled_model_by_trait.csv
  best_weekly_model_by_trait.csv
  cleaned_combined_data.csv
  pooled_model_summary.csv
  pooled_neste

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Check your browser downloads.


In [ ]:
# ============================================================
# HEMP PHENOME — DOWNLOAD ALL OUTPUTS
# Paste this as a NEW cell at the END of your Colab notebook.
# Run it after all previous cells have finished.
# It will produce every figure + CSV, zip them, and trigger
# a single browser download.
# ============================================================

import os, math, warnings, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.cross_decomposition import PLSRegression
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.inspection import permutation_importance
from google.colab import files

warnings.filterwarnings("ignore")

# ── Output folder ────────────────────────────────────────────────────────────
outdir = Path("/content/hemp_analysis_output_lean")
outdir.mkdir(parents=True, exist_ok=True)

# ── Styling helpers ──────────────────────────────────────────────────────────
sns.set_theme(style="white", font_scale=1.1)
model_order = ["Ridge", "PLSR", "RF", "GB", "SVR"]
trait_order  = ["Yield", "SPAD", "Height", "Canopy Cover"]
week_order   = ["45 DAT", "60 DAT", "75 DAT", "90 DAT"]
feature_cols = ["VARI", "TGI", "SIP2", "NDVI", "NDRE",
                "MCARI", "LCI", "GNDVI", "BNDVI", "MEAN_TD"]
target_map   = {
    "Yield_kg_per_ha":   "Yield",
    "Mean_SPAD":         "SPAD",
    "Mean_Height":       "Height",
    "Mean_Canopy_Cover": "Canopy Cover",
}

def _style_axes(ax):
    ax.grid(False)
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)

def get_xy(target_col):
    d = data[feature_cols + [target_col]].dropna().copy()
    return d[feature_cols], d[target_col]

# ── Save existing CSV results (already computed) ─────────────────────────────
print("Saving CV result tables …")
weekly_df.to_csv(outdir / "weekly_nested_cv_results.csv", index=False)
pooled_df.to_csv(outdir / "pooled_nested_cv_results.csv", index=False)

weekly_summary = (
    weekly_df.groupby(["Trait", "Model"], as_index=False)
    .agg(N=("N","mean"), R2_mean=("R2_LOOCV_Nested","mean"),
         RMSE_mean=("RMSE_LOOCV_Nested","mean"), MAE_mean=("MAE_LOOCV_Nested","mean"))
    .sort_values(["Trait","R2_mean"], ascending=[True,False])
)
pooled_summary = (
    pooled_df.groupby(["Trait", "Model"], as_index=False)
    .agg(N=("N","mean"), R2_mean=("R2_LOOCV_Nested","mean"),
         RMSE_mean=("RMSE_LOOCV_Nested","mean"), MAE_mean=("MAE_LOOCV_Nested","mean"))
    .sort_values(["Trait","R2_mean"], ascending=[True,False])
)
weekly_summary.to_csv(outdir / "weekly_model_summary.csv", index=False)
pooled_summary.to_csv(outdir / "pooled_model_summary.csv", index=False)

best_weekly = (weekly_df.sort_values(["Trait","R2_LOOCV_Nested"], ascending=[True,False])
               .groupby("Trait").head(1))
best_pooled = (pooled_df.sort_values(["Trait","R2_LOOCV_Nested"], ascending=[True,False])
               .groupby("Trait").head(1))
best_weekly.to_csv(outdir / "best_weekly_model_by_trait.csv", index=False)
best_pooled.to_csv(outdir / "best_pooled_model_by_trait.csv", index=False)

# ── Random Forest feature importance ─────────────────────────────────────────
print("Random Forest feature importance …")
rf_imp_rows = []
for tc, tn in target_map.items():
    if tc not in data.columns: continue
    X, y = get_xy(tc)
    if len(X) < 5: continue
    rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    for f, v in zip(feature_cols, rf.feature_importances_):
        rf_imp_rows.append([tn, f, v])
rf_imp   = pd.DataFrame(rf_imp_rows, columns=["Trait","Feature","Importance"])
rf_pivot = rf_imp.pivot_table(index="Feature", columns="Trait",
                               values="Importance", aggfunc="mean")
plt.figure(figsize=(10, 6))
ax = sns.heatmap(rf_pivot, cmap="Blues", annot=True, fmt=".3f",
                 linewidths=0.5, linecolor="white", cbar_kws={"label":"Importance"})
plt.title("Feature Importance (Random Forest)", fontweight="bold")
plt.xlabel(""); plt.ylabel("Feature"); _style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure9_RF_Feature_Importance.png", dpi=300, bbox_inches="tight")
plt.close()
rf_imp.to_csv(outdir / "Figure9_RF_Importance.csv", index=False)
rf_pivot.to_csv(outdir / "Figure9_RF_pivot.csv")

# ── Ridge coefficient importance ──────────────────────────────────────────────
print("Ridge coefficient importance …")
ridge_rows = []
for tc, tn in target_map.items():
    if tc not in data.columns: continue
    d = data[feature_cols + [tc]].dropna().copy()
    if len(d) < 5: continue
    X, y = d[feature_cols], d[tc]
    pipe = Pipeline([("scaler", StandardScaler()), ("model", Ridge(alpha=1.0))])
    pipe.fit(X, y)
    coef = np.abs(pipe.named_steps["model"].coef_)
    coef = coef / coef.sum() if coef.sum() != 0 else coef
    for f, v in zip(feature_cols, coef):
        ridge_rows.append([tn, f, v])
ridge_imp   = pd.DataFrame(ridge_rows, columns=["Trait","Feature","Importance"])
ridge_pivot = ridge_imp.pivot_table(index="Feature", columns="Trait",
                                     values="Importance", aggfunc="mean")
plt.figure(figsize=(10, 6))
ax = sns.heatmap(ridge_pivot, cmap="YlGnBu", annot=True, fmt=".3f",
                 linewidths=0.5, linecolor="white", cbar_kws={"label":"Importance"})
plt.title("Feature Importance (Ridge Regression)", fontweight="bold")
plt.xlabel(""); plt.ylabel("Feature"); _style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure10_Ridge_Feature_Importance.png", dpi=300, bbox_inches="tight")
plt.close()
ridge_imp.to_csv(outdir / "Figure10_Ridge_Importance.csv", index=False)
ridge_pivot.to_csv(outdir / "Figure10_Ridge_pivot.csv")

# ── Gradient Boosting SHAP importance ────────────────────────────────────────
print("Gradient Boosting SHAP importance …")
shap_rows = []
for tc, tn in target_map.items():
    if tc not in data.columns: continue
    d = data[feature_cols + [tc]].dropna().copy()
    if len(d) < 5: continue
    X, y = d[feature_cols], d[tc]
    gb = GradientBoostingRegressor(random_state=42)
    gb.fit(X, y)
    explainer = shap.TreeExplainer(gb)
    shap_vals = np.array(explainer.shap_values(X))
    imp = np.abs(shap_vals).mean(axis=0)
    imp = imp / imp.sum() if imp.sum() != 0 else imp
    for f, v in zip(feature_cols, imp):
        shap_rows.append([tn, f, v])
shap_imp   = pd.DataFrame(shap_rows, columns=["Trait","Feature","Importance"])
shap_pivot = shap_imp.pivot_table(index="Feature", columns="Trait",
                                   values="Importance", aggfunc="mean")
plt.figure(figsize=(10, 6))
ax = sns.heatmap(shap_pivot, cmap="magma", annot=True, fmt=".3f",
                 linewidths=0.5, linecolor="white", cbar_kws={"label":"Mean |SHAP|"})
plt.title("Feature Importance (Gradient Boosting SHAP)", fontweight="bold")
plt.xlabel(""); plt.ylabel("Feature"); _style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure11_GB_SHAP_Importance.png", dpi=300, bbox_inches="tight")
plt.close()
shap_imp.to_csv(outdir / "Figure11_GB_SHAP_Importance.csv", index=False)
shap_pivot.to_csv(outdir / "Figure11_GB_SHAP_pivot.csv")

# ── PLSR VIP importance ───────────────────────────────────────────────────────
print("PLSR VIP importance …")
def vip_scores(pls, X, y):
    T, W, Q = pls.x_scores_, pls.x_weights_, pls.y_loadings_
    p, h = W.shape
    s = np.diag(T.T @ T @ Q.T @ Q).reshape(h, -1)
    vip = np.sqrt(p * (W**2 @ s) / (s.sum() + 1e-12)).ravel()
    return vip

plsr_rows = []
for tc, tn in target_map.items():
    if tc not in data.columns: continue
    d = data[feature_cols + [tc]].dropna().copy()
    if len(d) < 5: continue
    X = d[feature_cols].values
    y = d[tc].values.reshape(-1, 1)
    ncomp = min(5, X.shape[1], max(1, X.shape[0] - 1))
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("model", PLSRegression(n_components=ncomp))])
    pipe.fit(X, y)
    vip = vip_scores(pipe.named_steps["model"], X, y)
    vip = vip / vip.sum() if vip.sum() != 0 else vip
    for f, v in zip(feature_cols, vip):
        plsr_rows.append([tn, f, v])
plsr_imp   = pd.DataFrame(plsr_rows, columns=["Trait","Feature","Importance"])
plsr_pivot = plsr_imp.pivot_table(index="Feature", columns="Trait",
                                   values="Importance", aggfunc="mean")
plt.figure(figsize=(10, 6))
ax = sns.heatmap(plsr_pivot, cmap="YlOrBr", annot=True, fmt=".3f",
                 linewidths=0.5, linecolor="white", cbar_kws={"label":"VIP"})
plt.title("PLSR Importance (VIP)", fontweight="bold")
plt.xlabel(""); plt.ylabel("Feature"); _style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure12_PLSR_VIP.png", dpi=300, bbox_inches="tight")
plt.close()
plsr_imp.to_csv(outdir / "Figure12_PLSR_VIP.csv", index=False)
plsr_pivot.to_csv(outdir / "Figure12_PLSR_VIP_pivot.csv")

# ── SVR permutation importance ────────────────────────────────────────────────
print("SVR permutation importance …")
perm_rows = []
for tc, tn in target_map.items():
    if tc not in data.columns: continue
    d = data[feature_cols + [tc]].dropna().copy()
    if len(d) < 5: continue
    X = d[feature_cols].values
    y = d[tc].values
    pipe = Pipeline([("scaler", StandardScaler()),
                     ("model", SVR(kernel="rbf", C=10, epsilon=0.1))])
    pipe.fit(X, y)
    perm = permutation_importance(pipe, X, y, n_repeats=20, random_state=42, scoring="r2")
    vals = perm.importances_mean
    vals = vals / np.sum(np.abs(vals)) if np.sum(np.abs(vals)) != 0 else vals
    for f, v in zip(feature_cols, vals):
        perm_rows.append([tn, f, v])
perm_imp   = pd.DataFrame(perm_rows, columns=["Trait","Feature","Importance"])
perm_pivot = perm_imp.pivot_table(index="Feature", columns="Trait",
                                   values="Importance", aggfunc="mean")
plt.figure(figsize=(10, 6))
ax = sns.heatmap(perm_pivot, cmap="coolwarm", center=0, annot=True, fmt=".3f",
                 linewidths=0.5, linecolor="white", cbar_kws={"label":"Permutation Importance"})
plt.title("SVR Permutation-Based Feature Importance", fontweight="bold")
plt.xlabel(""); plt.ylabel("Feature"); _style_axes(ax)
plt.tight_layout()
plt.savefig(outdir / "Figure13_SVR_Permutation_Importance.png", dpi=300, bbox_inches="tight")
plt.close()
perm_imp.to_csv(outdir / "Figure13_SVR_Permutation_Importance.csv", index=False)
perm_pivot.to_csv(outdir / "Figure13_SVR_Permutation_pivot.csv")

# ── ZIP everything and download ───────────────────────────────────────────────
print("\nZipping all outputs …")
zip_path = "/content/HEMP_all_outputs.zip"
all_files = sorted(outdir.iterdir())

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in all_files:
        if f.is_file():
            zf.write(f, arcname=f.name)

n = len([f for f in all_files if f.is_file()])
print(f"Zipped {n} files → {zip_path}")
print("\nFiles included:")
for f in sorted(all_files):
    if f.is_file():
        print(f"  {f.name}")

print("\nStarting download …")
files.download(zip_path)
print("Done! Check your browser downloads.")

Saving CV result tables …
Random Forest feature importance …
Ridge coefficient importance …
Gradient Boosting SHAP importance …
PLSR VIP importance …
SVR permutation importance …

Zipping all outputs …
Zipped 25 files → /content/HEMP_all_outputs.zip

Files included:
  Figure10_Ridge_Feature_Importance.png
  Figure10_Ridge_Importance.csv
  Figure10_Ridge_pivot.csv
  Figure11_GB_SHAP_Importance.csv
  Figure11_GB_SHAP_Importance.png
  Figure11_GB_SHAP_pivot.csv
  Figure12_PLSR_VIP.csv
  Figure12_PLSR_VIP.png
  Figure12_PLSR_VIP_pivot.csv
  Figure13_SVR_Permutation_Importance.csv
  Figure13_SVR_Permutation_Importance.png
  Figure13_SVR_Permutation_pivot.csv
  Figure7_Model_Performance_Heatmaps.png
  Figure8_Composite_Predictability.png
  Figure8_Composite_Predictability_Table.csv
  Figure9_RF_Feature_Importance.png
  Figure9_RF_Importance.csv
  Figure9_RF_pivot.csv
  best_pooled_model_by_trait.csv
  best_weekly_model_by_trait.csv
  cleaned_combined_data.csv
  pooled_model_summary.csv
  poo

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Done! Check your browser downloads.
